<a href="https://colab.research.google.com/github/raghad-cs/Esnad/blob/raghad-precedents/official_precedents_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas fastparquet pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import re

# 1. دالة تنظيف النص العربي
def normalize_arabic(text):
    if not isinstance(text, str):
        return ""
    # إزالة التشكيل والحركات
    text = re.sub(r'[\u064B-\u0652]', '', text)
    # توحيد أشكال الألف (أ، إ، آ -> ا)
    text = re.sub(r'[إأآا]', 'ا', text)
    # توحيد الألف المقصورة والياء (ى -> ي)
    text = re.sub(r'ى', 'ي', text)
    # توحيد التاء المربوطة والهاء (ة -> ه)
    text = re.sub(r'ة', 'ه', text)
    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 2. قراءة ملف الـ CSV مع تجربة الترميزات العربية (utf-8-sig أو utf-8 أو cp1256)
try:
    df = pd.read_csv('saudi_legal_cases.mvp.csv', encoding='utf-8-sig')
except UnicodeDecodeError:
    try:
        df = pd.read_csv('saudi_legal_cases.mvp.csv', encoding='cp1256')
    except UnicodeDecodeError:
        df = pd.read_csv('saudi_legal_cases.mvp.csv', encoding='utf-16')

# 3. دمج الخانات في نص واحد وتنظيفه
df['clean_text'] = df.apply(
    lambda row: normalize_arabic(
        f"عنوان القضية: {row.get('title', '')}. "
        f"الوقائع: {row.get('facts', '')}. "
        f"طلبات المدعي: {row.get('plaintiff_claims', '')}. "
        f"رد المدعى عليه: {row.get('defendant_response', '')}. "
        f"السبب الشرعي: {row.get('legal_reasoning', '')}. "
        f"الملخص: {row.get('summary', '')}."
    ), axis=1
)

# 4. طباعة النتيجة والتأكد
print(f"تمت معالجة {len(df)} حكم بنجاح!")
print("\nعينة من النص المنظف للحكم الأول:")
print(df['clean_text'].iloc[0])

# 5. حفظ البيانات بصيغة parquet
df.to_parquet('precedents_processed.parquet')
print("\nتم حفظ الملف باسم precedents_processed.parquet بنجاح!")

تمت معالجة 20 حكم بنجاح!

عينة من النص المنظف للحكم الأول:
عنوان القضيه: اثبات بيع عقار بدون صك. الوقائع: شراء ارض زراعيه من شخص افقده حادث الوعي ومعه سندات بيده. طلبات المدعي: اثبات البيع للمشتريين. رد المدعي عليه: المقر بشرعيه الولايه اقر بصحه الدعوي والشراء. السبب الشرعي: الاستناد الي السندات المكتوبه واليمين المكمله والمادتين (179 و258/2). الملخص: ثبوت بيع ارض زراعيه غير مسجله بصك استنادا لسندات بخط البائع ويمين مكمله مع النص علي عدم افادتها للتملك.

تم حفظ الملف باسم precedents_processed.parquet بنجاح!


In [5]:
%%writefile prepare_precedents.py
import pandas as pd
import re

def normalize_arabic(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u064B-\u0652]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def prepare_data():
    df = pd.read_csv('saudi_legal_cases.mvp.csv')
    df['clean_text'] = df.apply(
        lambda row: normalize_arabic(
            f"عنوان القضية: {row.get('title', '')}. "
            f"الوقائع: {row.get('facts', '')}. "
            f"طلبات المدعي: {row.get('plaintiff_claims', '')}. "
            f"رد المدعى عليه: {row.get('defendant_response', '')}. "
            f"السبب الشرعي: {row.get('legal_reasoning', '')}. "
            f"الملخص: {row.get('summary', '')}."
        ), axis=1
    )
    df.to_parquet('precedents_processed.parquet')
    print("تم تجهيز وتنظيف داتا السوابق بنجاح!")

if __name__ == "__main__":
    prepare_data()

Overwriting prepare_precedents.py


In [8]:
%%writefile prepare_precedents.py
import pandas as pd
import re

def normalize_arabic(text):
    if not isinstance(text, str):
        return ""
    # إزالة التشكيل والحركات
    text = re.sub(r'[\u064B-\u0652]', '', text)
    # توحيد أشكال الألف
    text = re.sub(r'[إأآا]', 'ا', text)
    # توحيد الألف المقصورة والياء
    text = re.sub(r'ى', 'ي', text)
    # توحيد التاء المربوطة والهاء
    text = re.sub(r'ة', 'ه', text)
    # إزالة المسافات الزائدة
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def prepare_data():
    file_path = 'saudi_legal_cases.mvp.csv'

    # محاولة قراءة الملف بتجربة ترميزات متعددة لمنع خطأ الـ UnicodeDecodeError
    encodings = ['utf-8', 'utf-8-sig', 'cp1256', 'windows-1256', 'latin1']
    df = None

    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc)
            print(f" تم قراءة الملف بنجاح باستخدام ترميز: {enc}")
            break
        except (UnicodeDecodeError, Exception):
            continue

    if df is None:
        raise ValueError(" تعذر قراءة الملف بالترميزات المتاحة. يرجى التأكد من سلامة الملف.")

    # دمج وتنظيف الخانات في نص واحد
    df['clean_text'] = df.apply(
        lambda row: normalize_arabic(
            f"عنوان القضية: {row.get('title', '')}. "
            f"الوقائع: {row.get('facts', '')}. "
            f"طلبات المدعي: {row.get('plaintiff_claims', '')}. "
            f"رد المدعى عليه: {row.get('defendant_response', '')}. "
            f"السبب الشرعي: {row.get('legal_reasoning', '')}. "
            f"الملخص: {row.get('summary', '')}."
        ), axis=1
    )

    # حفظ الملف الناتج بصيغة parquet
    df.to_parquet('precedents_processed.parquet')
    print(f"✅ تم معالجة {len(df)} أحكام/سوابق بنجاح!")
    print("✅ تم حفظ الملف المعالج باسم: precedents_processed.parquet")

if __name__ == "__main__":
    prepare_data()

Overwriting prepare_precedents.py


In [9]:
!python prepare_precedents.py

 تم قراءة الملف بنجاح باستخدام ترميز: cp1256
✅ تم معالجة 20 أحكام/سوابق بنجاح!
✅ تم حفظ الملف المعالج باسم: precedents_processed.parquet


In [10]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.1 MB/s eta 0:00:00


In [11]:
%%writefile build_precedents_index.py
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# الإعدادات الموحدة المعتمدة في المشروع
MODEL_NAME = "BAAI/bge-m3"

def build_index():
    print("⏳ جاري قراءة ملف precedents_processed.parquet...")
    df = pd.read_parquet('precedents_processed.parquet')

    print(f"⏳ جاري تحميل نموذج الـ Embeddings ({MODEL_NAME})...")
    model = SentenceTransformer(MODEL_NAME)

    print("⏳ جاري تحويل النصوص القضائية إلى Embeddings...")
    embeddings = model.encode(
        df['clean_text'].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True
    )
    embeddings = np.array(embeddings).astype('float32')

    print("⏳ جاري إنشاء فهرس FAISS (IndexFlatIP)...")
    dimension = embeddings.shape[1] # أبعاد النموذج (1024)
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    # حفظ الفهرس والـ Metadata
    faiss.write_index(index, 'precedents.index')
    df.to_parquet('precedents_metadata.parquet')

    print("✅ تم إنشاء الفهرس بنجاح وحفظه باسم: precedents.index")
    print("✅ تم حفظ البيانات الوصفية باسم: precedents_metadata.parquet")

if __name__ == "__main__":
    build_index()

Writing build_precedents_index.py


In [12]:
!python build_precedents_index.py

⏳ جاري قراءة ملف precedents_processed.parquet...
⏳ جاري تحميل نموذج الـ Embeddings (BAAI/bge-m3)...
modules.json: 100% 349/349 [00:00<00:00, 1.02MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 314kB/s]
README.md: 100% 15.8k/15.8k [00:00<00:00, 21.3MB/s]
sentence_bert_config.json: 100% 54.0/54.0 [00:00<00:00, 198kB/s]
config.json: 100% 687/687 [00:00<00:00, 1.87MB/s]

pytorch_model.bin: downloading bytes:  14% 326M/2.27G [00:02<00:12, 152MB/s, 27.6MB/s  ]
pytorch_model.bin: downloading bytes:  18% 408M/2.27G [00:02<00:10, 177MB/s, 34.4MB/s  ]
pytorch_model.bin: downloading bytes:  19% 431M/2.27G [00:03<00:14, 129MB/s, 35.9MB/s  ]
pytorch_model.bin: downloading bytes:  29% 667M/2.27G [00:04<00:11, 142MB/s, 51.6MB/s  ]
pytorch_model.bin: downloading bytes:  38% 873M/2.27G [00:05<00:09, 145MB/s, 64.0MB/s  ]
pytorch_model.bin: downloading bytes:  57% 1.31G/2.27G [00:09<00:18, 51.3MB/s, 74.9MB/s  ]
pytorch_model.bin: downloading bytes:  67% 1.52G/2.27G [00:15<00:23, 32.5M

In [13]:
%%writefile search_precedents.py
import faiss
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# الإعدادات الموحدة
MODEL_NAME = "BAAI/bge-m3"

class PrecedentsSearchEngine:
    def __init__(self):
        print("⏳ جاري تحميل نموذج البحث وفهرس FAISS...")
        self.model = SentenceTransformer(MODEL_NAME)
        self.index = faiss.read_index('precedents.index')
        self.metadata = pd.read_parquet('precedents_metadata.parquet')
        print("✅ تم تحميل المحرك وفهرس السوابق بنجاح!")

    def search(self, query, top_k=3):
        # 1. تحويل نص البحث إلى Embedding ومعيرته
        query_vector = self.model.encode([query], normalize_embeddings=True)
        query_vector = np.array(query_vector).astype('float32')

        # 2. البحث في FAISS عن أفضل top_k نتائج
        scores, indices = self.index.search(query_vector, top_k)

        # 3. تجميع النتائج
        results = []
        for idx, score in zip(indices[0], scores[0]):
            if idx != -1:
                row = self.metadata.iloc[idx].to_dict()
                row['similarity_score'] = float(score)
                results.append(row)
        return results

# تجربة سريعة للبحث عند تشغيل الملف المباشر
if __name__ == "__main__":
    search_engine = PrecedentsSearchEngine()

    # تجربة استفسار تجريبي
    test_query = "طالب بتسليم باقي ثمن سيارة مباعة بالتقسيط والمشتري ما سدد"
    print(f"\n🔍 تجربة البحث عن: '{test_query}'\n" + "-"*50)

    results = search_engine.search(test_query, top_k=3)

    for i, res in enumerate(results, 1):
        print(f"\n📌 السابقة رقم ({i}): {res.get('title', 'بدون عنوان')}")
        print(f"📊 نسبة التشابه: {res['similarity_score']:.4f}")
        print(f"🏛️ المحكمة: {res.get('court', 'N/A')}")
        print(f"📝 الملخص: {res.get('summary', 'N/A')}")

Writing search_precedents.py


In [20]:
!python search_precedents.py

⏳ جاري تحميل نموذج البحث وفهرس FAISS...
Loading weights: 100% 391/391 [00:00<00:00, 13055.76it/s]
✅ تم تحميل المحرك وفهرس السوابق بنجاح!

🔍 تجربة البحث عن: 'طالب بتسليم باقي ثمن سيارة مباعة بالتقسيط والمشتري ما سدد'
--------------------------------------------------

📌 السابقة رقم (1): المطالبة بباقي ثمن سيارة بالتقسيط من ورثة المشتري
📊 نسبة التشابه: 0.7017
🏛️ المحكمة: المحكمة العامة بالرياض
📝 الملخص: ثبوت باقي ثمن سيارة تقسيط بذمة متوفى وإلزام الورثة بسداده من التركة قبل توزيع الإرث

📌 السابقة رقم (2): المطالبة بباقي ثمن سيارة بالتقسيط ونكول المشتري
📊 نسبة التشابه: 0.6884
🏛️ المحكمة: المحكمة العامة بالطائف
📝 الملخص: إلزام مشتري سيارة بالتقسيط بدفع الباقي غيابياً استناداً للكمبيالات ونكوله عن الجواب ويمين المدعي

📌 السابقة رقم (3): المطالبة بثمن سيارتين بعد الاستمهال وعجز البينة
📊 نسبة التشابه: 0.6309
🏛️ المحكمة: المحكمة العامة بمكة المكرمة
📝 الملخص: إلزام مشتري سيارتين بدفع كامل الثمن بعد الإقرار بالشراء وعجزه عن إثبات ادعاء السداد ورفضه يمين المدعي


In [24]:
%%writefile search_engine.py
import faiss
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"

class LegalSearchEngine:
    def __init__(self):
        print("⏳ جاري تحميل نموذج BGE-M3 الموحد...")
        self.model = SentenceTransformer(MODEL_NAME)

        # 1. تحميل فهرس السوابق التجاري
        try:
            self.precedents_index = faiss.read_index('precedents.index')
            self.precedents_meta = pd.read_parquet('precedents_metadata.parquet')
            print("✅ تم تحميل فهرس السوابق التجاري بنجاح.")
        except Exception as e:
            print(f"⚠️ تنبيه: تعذر تحميل فهرس السوابق ({e})")
            self.precedents_index = None

        # 2. تحميل فهرس ALARB
        try:
            self.judgments_index = faiss.read_index('judgments.index')
            self.judgments_meta = pd.read_parquet('judgments_metadata.parquet')
            print("✅ تم تحميل فهرس أحكام ALARB بنجاح.")
        except Exception as e:
            print(f"ℹ️ ملاحظة: فهرس ALARB غير موجود حالياً وسيتوفر عند دمج ال.")
            self.judgments_index = None

    def search_all(self, query, top_k_precedents=3, top_k_judgments=5):
        # تحويل الاستفسار لـ Query Embedding مرة واحدة فقط
        query_vector = self.model.encode([query], normalize_embeddings=True)
        query_vector = np.array(query_vector).astype('float32')

        results = {
            "precedents": [],
            "judgments": []
        }

        # البحث في فهرس السوابق
        if self.precedents_index is not None:
            scores, indices = self.precedents_index.search(query_vector, top_k_precedents)
            for idx, score in zip(indices[0], scores[0]):
                if idx != -1:
                    row = self.precedents_meta.iloc[idx].to_dict()
                    row['score'] = float(score)
                    results["precedents"].append(row)

        # البحث في فهرس ALARB
        if self.judgments_index is not None:
            scores, indices = self.judgments_index.search(query_vector, top_k_judgments)
            for idx, score in zip(indices[0], scores[0]):
                if idx != -1:
                    row = self.judgments_meta.iloc[idx].to_dict()
                    row['score'] = float(score)
                    results["judgments"].append(row)

        return results

if __name__ == "__main__":
    # تجربة المحرك المدمج
    engine = LegalSearchEngine()
    test_res = engine.search_all("طالب بدفع باقي ثمن سيارة تقسيط وما سدد المشتري")
    print(f"\n✅ عينة من نتائج البحث الموحد (السوابق): {len(test_res['precedents'])} نتائج.")

Overwriting search_engine.py


In [25]:
!python search_engine.py

⏳ جاري تحميل نموذج BGE-M3 الموحد...
Loading weights: 100% 391/391 [00:00<00:00, 18738.69it/s]
✅ تم تحميل فهرس السوابق التجاري بنجاح.
ℹ️ ملاحظة: فهرس ALARB غير موجود حالياً وسيتوفر عند دمج ال.

✅ عينة من نتائج البحث الموحد (السوابق): 3 نتائج.
